# 🔧 Shared Utilities
Run all cells in this notebook **once** before opening any model notebook.
These helpers are used by Models 1–6 and Phase 3.


## 0. Mount Google Drive
Required since this notebook reads/writes files under `/content/drive/MyDrive/...`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install scikit-learn openpyxl -q

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from typing import Literal

BAND_NAMES = ["Delta", "Theta", "Alpha", "Beta", "Gamma"]

In [ ]:
def load_dataset(path):
    df = pd.read_excel(path) if path.endswith(".xlsx") else pd.read_csv(path)
    float_cols = df.select_dtypes(include=["float64"]).columns
    df[float_cols] = df[float_cols].astype(np.float32)
    return df

def get_band_columns(df):
    return {b: [c for c in df.columns if c.startswith(f"{b}_")] for b in BAND_NAMES}

def prepare_tensors(df, mode: Literal["2d", "seq", "flat"] = "seq"):
    """
    mode='2d'  -> (N, n_ch, N_BANDS, 1)   for CNN-2D
    mode='seq' -> (N, N_BANDS, n_ch)       for LSTM / Conformer / hybrids
    mode='flat'-> (N, N_BANDS * n_ch)      for SVM baseline
    """
    band_cols = get_band_columns(df)
    band_matrices = [df[band_cols[b]].values for b in BAND_NAMES]
    X_seq = np.stack(band_matrices, axis=1)   # (N, 5, n_ch)
    y     = df["label"].values.astype(int)

    if mode == "2d":
        X = X_seq[..., np.newaxis]            # (N, 5, n_ch, 1)
        return X.transpose(0, 2, 1, 3), y     # (N, n_ch, 5, 1)
    elif mode == "seq":
        return X_seq, y
    elif mode == "flat":
        return X_seq.reshape(len(df), -1), y
    raise ValueError(f"Unknown mode: {mode}")

def normalize_tensors(Xtr, Xv, Xte):
    """Fit mean/std on TRAIN split only (per-feature), apply to all splits.
    Prevents unnormalized band-power features (which can span orders of
    magnitude) from destabilizing BatchNorm / causing AUC-stuck-at-0.5 collapse."""
    mean = Xtr.mean(axis=0, keepdims=True)
    std  = Xtr.std(axis=0, keepdims=True)
    Xtr  = (Xtr - mean) / (std + 1e-8)
    Xv   = (Xv  - mean) / (std + 1e-8)
    Xte  = (Xte - mean) / (std + 1e-8)
    return Xtr, Xv, Xte

def compute_output_bias(y_train):
    """Prior-informed bias init for the final sigmoid layer, given severe class
    imbalance (e.g. ~0.2% seizure ratio). Pass to a model's output Dense layer's
    bias_initializer so training doesn't start by fighting a bad default prior."""
    import numpy as np
    n1, total = y_train.sum(), len(y_train)
    return float(np.log(n1 / (total - n1)))


In [ ]:
def evaluate_with_svm(feat_train, y_train, feat_test, y_test, kernel="rbf"):
    sc = StandardScaler()
    X_tr = sc.fit_transform(feat_train)
    X_te = sc.transform(feat_test)
    svm  = SVC(kernel=kernel, C=1.0, gamma="scale",
               class_weight="balanced", random_state=42)
    svm.fit(X_tr, y_train)
    preds = svm.predict(X_te)

    tn, fp, fn, tp = confusion_matrix(y_test, preds, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0   # recall on Seizure class
    specificity = tn / (tn + fp) if (tn + fp) else 0.0   # recall on No-Seizure class

    return {
        "accuracy":    round(accuracy_score(y_test, preds), 4),
        "sensitivity": round(sensitivity, 4),
        "specificity": round(specificity, 4),
        "f1_macro":    round(f1_score(y_test, preds, average="macro"), 4),
        "f1_weighted": round(f1_score(y_test, preds, average="weighted"), 4),
        "report":      classification_report(y_test, preds,
                           labels=[0, 1], target_names=["No Seizure", "Seizure"]),
    }

def get_optimizers(lr=1e-3):
    from tensorflow.keras import optimizers
    # clipnorm=1.0: with severe class imbalance (class_weight ~500x on the
    # minority class), a single misclassified positive can produce a huge
    # weighted gradient -- clipping prevents that from destabilizing training.
    return {
        "Adam":  optimizers.Adam(learning_rate=lr, clipnorm=1.0),
        "Nadam": optimizers.Nadam(learning_rate=lr, clipnorm=1.0),
        "AdamW": optimizers.AdamW(learning_rate=lr, weight_decay=1e-4, clipnorm=1.0),
    }

def save_results(row, path="/content/drive/MyDrive/epilepsy_outputs/results.csv"):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df = pd.DataFrame([row])
    df.to_csv(path, mode="a", header=not os.path.exists(path), index=False)
    print("Saved:", row)

print(" Utilities loaded.")


 Utilities loaded.
